In [2]:
import torch 
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

In [3]:
# start with a Hugging Face tokenizer / BPE와 subword tokenization

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

text = "He wanted to eat Chinese food and behaveed uncharracteristically."

tokens = tokenizer.tokenize(text)

ids = tokenizer.encode(text)

print(tokens)
print(ids)

for tk_id in ids:
    print(
        tk_id,
        tokenizer.decode([tk_id])
    )

'''
1. 서브 워드 분해 ( 오타단어와 긴 파생어 처리 )

2. 특수 기호 Ġ의 의미 ( 단어 앞에 붙은 공백을 의미 )
'''

    

['He', 'Ġwanted', 'Ġto', 'Ġeat', 'ĠChinese', 'Ġfood', 'Ġand', 'Ġbehave', 'ed', 'Ġun', 'ch', 'arr', 'acter', 'istically', '.']
[1519, 4829, 311, 8180, 8453, 3607, 323, 35692, 291, 650, 331, 1118, 2481, 37110, 13]
1519 He
4829  wanted
311  to
8180  eat
8453  Chinese
3607  food
323  and
35692  behave
291 ed
650  un
331 ch
1118 arr
2481 acter
37110 istically
13 .


'\n1. 서브 워드 분해 ( 오타단어와 긴 파생어 처리 )\n\n2. 특수 기호 Ġ의 의미 ( 단어 앞에 붙은 공백을 의미 )\n'

In [4]:
# 입력 텐서 만들기

inputs = tokenizer(
    text,
    return_tensors="pt"
)

print(inputs["input_ids"])
print(inputs["input_ids"].shape, "\n\n")

print(inputs["attention_mask"]) # 모델이 어떤 토큰에 주의를 기울여야 하는 지 알려주는 마스크 ( 1은 실제 유효한 토큰, 0은 길이 맞춤용 패딩 토큰 )
print(inputs["attention_mask"].shape, "\n\n")

print(inputs)

tensor([[ 1519,  4829,   311,  8180,  8453,  3607,   323, 35692,   291,   650,
           331,  1118,  2481, 37110,    13]])
torch.Size([1, 15]) 


tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
torch.Size([1, 15]) 


{'input_ids': tensor([[ 1519,  4829,   311,  8180,  8453,  3607,   323, 35692,   291,   650,
           331,  1118,  2481, 37110,    13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [5]:
# Load the Real Causal model

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device_map="auto"
)
model.eval()

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

c:\Users\DM501TGA\anaconda3\envs\llm_lab01\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DM501TGA\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [ ]:
inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}


# find out the logits

with torch.no_grad():
    outputs = model(**inputs)
    
logits = outputs.logits

print("input_ids:", inputs["input_ids"].shape)
print("logits:", logits.shape)

'''
logits : (batch_size, sequence_length, vocab_size)
'''

input_ids: torch.Size([1, 15])
logits: torch.Size([1, 15, 151936])


In [7]:
# Extract the next_logit
next_logits = logits[0, -1]
print("next_logits:", next_logits.shape)

# convert logits to probabilities and topK
probs = torch.softmax(next_logits, dim=-1)
top_probs, top_ids = torch.topk(probs, k = 10)
for prob, tk_id in zip(top_probs, top_ids):
    token = tokenizer.decode([tk_id.item()])
    print(repr(token), f"{prob.item():4f}")

next_logits: torch.Size([151936])
' ' 0.632812
' He' 0.058838
' What' 0.029541
' A' 0.020264
' Choose' 0.017944
' Which' 0.013123
' The' 0.011597
' This' 0.009583
' It' 0.009033
' How' 0.007935


In [8]:
# Manually generate One Token
next_id = top_ids[0]

new_ids = torch.cat(
    [
        inputs["input_ids"],
        next_id.view(1,1)
    ],
    dim=-1
)
print(tokenizer.decode(new_ids[0]))

He wanted to eat Chinese food and behaveed uncharracteristically. 
